# **200m Career Data**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.figure_factory as ff
%matplotlib inline
from datetime import datetime
import re
from nssstats.plots import std_plot
from nssstats.plots import iqr_plot
from nssstats.plots import quadrant_plot, half_plot
from nssstats.cm import cm_analysis
from ipywidgets import interact, FloatSlider
from sklearn.model_selection import train_test_split
from scipy.stats import probplot
from scipy.stats import t, sem
from scipy.stats import chi2
from statsmodels.stats.proportion import proportion_confint
import statsmodels.formula.api as sm

In [ ]:
sprinters = pd.read_csv("../data/Worlds_Fastest_Sprinters_Stats.csv")

# **Data** **Basics**

In [ ]:
sprinters.head()

In [ ]:
#Helps with data cleaning
sprinters['Athlete'] = sprinters['Athlete'].str.strip()

In [ ]:
#Replace for data cleaning
sprinters['Athlete'] = sprinters['Athlete'].replace({'Christophe Lamaitre': 'Christophe Lemaitre'})

In [ ]:
sprinters.info

In [ ]:
sprinters.shape

In [ ]:
print(sprinters.dtypes)

In [ ]:
sprinters.describe()

In [ ]:
sprinters.corr

In [ ]:
sprinters.isnull().sum()

# **General** **EDA**

Let's make a column for the total PR time

In [ ]:
sprinters['Total_Time_PRs'] = round(sprinters['100_PR'] + sprinters['200_PR'] + sprinters['400_PR'],2)
sprinters.head()

Let's make a column for the total career average time

In [ ]:
sprinters['Total_Time_SB_Avg'] = round(sprinters['Avg_Season_Best_100m'] + sprinters['Avg_Season_Best_200m'] + sprinters['Avg_Season_Best_400m'],2)
sprinters.head()

Let's make a column for the actual age of the athletes

In [ ]:
# Ensure the DOB column is in datetime format
sprinters['DOB'] = pd.to_datetime(sprinters['DOB'], errors='coerce')  # Coerce will handle invalid dates as NaT

# Get the current year
current_year = datetime.now().year

# Calculate the age by subtracting the birth year from the current year
sprinters['Age'] = current_year - sprinters['DOB'].dt.year

# Display the updated DataFrame with the new 'Age' column
print(sprinters[['DOB', 'Age']].head())

In [ ]:
Sprinter_DOB = sprinters[['Athlete','DOB']]
Sprinter_DOB

**Let's add the honors data**

**Overall honors data can be found inthe 100m_Career_Analysis_Short_Summary notebook**

In [ ]:
honors = pd.read_csv("../data/Sprinter_Honors.csv")
honors.head()

In [ ]:
#Helps with data cleaning
honors['Athlete'] = honors['Athlete'].str.strip()

In [ ]:
#For data cleaning purposes
honors['Athlete'] = honors['Athlete'].replace({'Christophe Lamaitre': 'Christophe Lemaitre'})

In [ ]:
Honors_200m = honors[honors['Event'] == '200m']
Honors_200m.head(3)

In [ ]:
# ============================================================
# Helper function for column naming
# ============================================================
def make_col_prefix(name):
    name = name.replace('(', '').replace(')', '')
    name = name.replace('-', ' ')
    return '_'.join(name.split())

# ============================================================
# Base medal counts by place (gold, silver, bronze)
# ============================================================
medals_200m = Honors_200m[Honors_200m['Place'].isin([1, 2, 3])].groupby(['Athlete', 'Place']).size().unstack(fill_value=0)
for place in [1, 2, 3]:
    if place not in medals_200m.columns:
        medals_200m[place] = 0
medals_200m = medals_200m.rename(columns={1: 'gold', 2: 'silver', 3: 'bronze'})
medals_200m['total_medals'] = medals_200m[['gold', 'silver', 'bronze']].sum(axis=1)
medals_200m = medals_200m.reset_index()


# ============================================================
# Senior vs Junior medals (totals + gold/silver/bronze)
# ============================================================
senior_medals = Honors_200m[(Honors_200m['Place'].isin([1, 2, 3])) & (Honors_200m['Competition_Level'] == 'Senior')].groupby('Athlete').size().reset_index(name='Total_Senior_Medals')
junior_medals = Honors_200m[(Honors_200m['Place'].isin([1, 2, 3])) & (Honors_200m['Competition_Level'] == 'Junior')].groupby('Athlete').size().reset_index(name='Total_Junior_Medals')

medals_200m = medals_200m.merge(senior_medals, on='Athlete', how='left')
medals_200m = medals_200m.merge(junior_medals, on='Athlete', how='left')

medals_200m['Total_Senior_Medals'] = medals_200m['Total_Senior_Medals'].fillna(0).astype(int)
medals_200m['Total_Junior_Medals'] = medals_200m['Total_Junior_Medals'].fillna(0).astype(int)

senior_gsb = Honors_200m[(Honors_200m['Place'].isin([1, 2, 3])) & (Honors_200m['Competition_Level'] == 'Senior')].groupby(['Athlete', 'Place']).size().unstack(fill_value=0)
for place in [1, 2, 3]:
    if place not in senior_gsb.columns:
        senior_gsb[place] = 0
senior_gsb = senior_gsb.rename(columns={1: 'Senior_gold', 2: 'Senior_silver', 3: 'Senior_bronze'}).reset_index()

junior_gsb = Honors_200m[(Honors_200m['Place'].isin([1, 2, 3])) & (Honors_200m['Competition_Level'] == 'Junior')].groupby(['Athlete', 'Place']).size().unstack(fill_value=0)
for place in [1, 2, 3]:
    if place not in junior_gsb.columns:
        junior_gsb[place] = 0
junior_gsb = junior_gsb.rename(columns={1: 'Junior_gold', 2: 'Junior_silver', 3: 'Junior_bronze'}).reset_index()

medals_200m = medals_200m.merge(senior_gsb, on='Athlete', how='left')
medals_200m = medals_200m.merge(junior_gsb, on='Athlete', how='left')

for col in ['Senior_gold', 'Senior_silver', 'Senior_bronze', 'Junior_gold', 'Junior_silver', 'Junior_bronze']:
    medals_200m[col] = medals_200m[col].fillna(0).astype(int)




# ============================================================
# Championship_Type breakdown (totals + gold/silver/bronze + boolean flags),
# includes Olympic Games
# ============================================================
championship_types = [
    'Olympic Games', 'World Championships', 'World Indoor Championships', 'World (Continental) Cup',
    'World Athletics Final', 'World Athletics Relays', 'World U20 Championships','World U18 Championships',
    'Diamond League','Diamond League Final','Golden League','National Championships','National Indoor Championships',
    'African Championships','African U20 Championships','NACAC Championships', 'NACAC U23 Championships',
    'European Championships', 'European Indoor Championships', 'European Team Championships',
    'European Team Championships 1st League', 'European Team Championships 3rd League',
    'South American Championships','South American U23 Championships', 'Asian Championships', 'Asian Indoor Championships',
    'All-African Games', 'Commonwealth Games', 'CAC Championships', 'CAC Games', 'Pan American Games', 'Pan American U20 Championships',
    'Francophone Games','Mediterranean Games', 'Balkan Championships','Pan-Arab Games', 'Asian Games',
    'Ibero-American Championships', 'Grand Prix Final', 'World University Games', 'World Military Games',
    'NCAA Championships','NCAA Indoor Championships', 'Youth Olympic Games'
    
]

for champ in championship_types:
    prefix = make_col_prefix(champ)

    total = Honors_200m[(Honors_200m['Place'].isin([1, 2, 3])) & (Honors_200m['Championship_Type'] == champ)].groupby('Athlete').size().reset_index(name=f'Total_{prefix}_Medals')
    medals_200m = medals_200m.merge(total, on='Athlete', how='left')
    medals_200m[f'Total_{prefix}_Medals'] = medals_200m[f'Total_{prefix}_Medals'].fillna(0).astype(int)

    gsb = Honors_200m[(Honors_200m['Place'].isin([1, 2, 3])) & (Honors_200m['Championship_Type'] == champ)].groupby(['Athlete', 'Place']).size().unstack(fill_value=0)
    for place in [1, 2, 3]:
        if place not in gsb.columns:
            gsb[place] = 0
    gsb = gsb.rename(columns={1: f'{prefix}_gold', 2: f'{prefix}_silver', 3: f'{prefix}_bronze'})
    gsb = gsb.reset_index()

    medals_200m = medals_200m.merge(gsb, on='Athlete', how='left')
    for suffix in ['gold', 'silver', 'bronze']:
        medals_200m[f'{prefix}_{suffix}'] = medals_200m[f'{prefix}_{suffix}'].fillna(0).astype(int)

    # Boolean medalist flags
    medals_200m[f'{prefix}_Medalist'] = medals_200m[f'Total_{prefix}_Medals'] > 0
    medals_200m[f'{prefix}_Champion'] = medals_200m[f'{prefix}_gold'] > 0
    medals_200m[f'{prefix}_Silver_Medalist'] = medals_200m[f'{prefix}_silver'] > 0
    medals_200m[f'{prefix}_Bronze_Medalist'] = medals_200m[f'{prefix}_bronze'] > 0

# ============================================================
# Event breakdown (totals + gold/silver/bronze only, no booleans)
# ============================================================
event_types = Honors_200m['Event'].dropna().unique().tolist()

for event in event_types:
    prefix = make_col_prefix(event)

    total = Honors_200m[(Honors_200m['Place'].isin([1, 2, 3])) & (Honors_200m['Event'] == event)].groupby('Athlete').size().reset_index(name=f'Total_{prefix}_Medals')
    medals_200m = medals_200m.merge(total, on='Athlete', how='left')
    medals_200m[f'Total_{prefix}_Medals'] = medals_200m[f'Total_{prefix}_Medals'].fillna(0).astype(int)

    gsb = Honors_200m[(Honors_200m['Place'].isin([1, 2, 3])) & (Honors_200m['Event'] == event)].groupby(['Athlete', 'Place']).size().unstack(fill_value=0)
    for place in [1, 2, 3]:
        if place not in gsb.columns:
            gsb[place] = 0
    gsb = gsb.rename(columns={1: f'{prefix}_gold', 2: f'{prefix}_silver', 3: f'{prefix}_bronze'})
    gsb = gsb.reset_index()

    medals_200m = medals_200m.merge(gsb, on='Athlete', how='left')
    for suffix in ['gold', 'silver', 'bronze']:
        medals_200m[f'{prefix}_{suffix}'] = medals_200m[f'{prefix}_{suffix}'].fillna(0).astype(int)

# ============================================================
# Event breakdown by Competition_Level (Senior & Junior)
# totals + gold/silver/bronze only, no booleans
# ============================================================
competition_levels = ['Senior', 'Junior']

for event in event_types:
    event_prefix = make_col_prefix(event)

    for level in competition_levels:
        prefix = f'{level}_{event_prefix}'

        total = Honors_200m[
            (Honors_200m['Place'].isin([1, 2, 3])) &
            (Honors_200m['Event'] == event) &
            (Honors_200m['Competition_Level'] == level)
        ].groupby('Athlete').size().reset_index(name=f'Total_{prefix}_Medals')

        medals_200m = medals_200m.merge(total, on='Athlete', how='left')
        medals_200m[f'Total_{prefix}_Medals'] = medals_200m[f'Total_{prefix}_Medals'].fillna(0).astype(int)

        gsb = Honors_200m[
            (Honors_200m['Place'].isin([1, 2, 3])) &
            (Honors_200m['Event'] == event) &
            (Honors_200m['Competition_Level'] == level)
        ].groupby(['Athlete', 'Place']).size().unstack(fill_value=0)

        for place in [1, 2, 3]:
            if place not in gsb.columns:
                gsb[place] = 0
        gsb = gsb.rename(columns={1: f'{prefix}_gold', 2: f'{prefix}_silver', 3: f'{prefix}_bronze'})
        gsb = gsb.reset_index()

        medals_200m = medals_200m.merge(gsb, on='Athlete', how='left')
        for suffix in ['gold', 'silver', 'bronze']:
            medals_200m[f'{prefix}_{suffix}'] = medals_200m[f'{prefix}_{suffix}'].fillna(0).astype(int)


# ============================================================
# Medal Score — based on World Athletics Placing Score rubric
# ============================================================
PLACING_SCORES_200m = {
    #        1st   2nd   3rd
    "OW": {1: 350, 2: 310, 3: 280},
    "DF": {1: 240, 2: 210, 3: 185},
    "GW": {1: 200, 2: 170, 3: 150},
    "GL": {1: 170, 2: 145, 3: 130},
    "A":  {1: 140, 2: 120, 3: 110},
    "B":  {1: 100, 2:  80, 3:  70},
    "C":  {1:  60, 2:  50, 3:  45},
    "D":  {1:  40, 2:  35, 3:  30},
    "E":  {1:  25, 2:  21, 3:  18},
    "F":  {1:  15, 2:  10, 3:   5},
}

def placing_score(category: str, place: int, default: float = 0.0) -> float:
    """
    Return the Placing Score for a given category and finishing position.
    Only places 1-3 are covered here; anything else returns `default`.
    """
    cat = category.upper().strip()
    if cat not in PLACING_SCORES_200m:
        raise ValueError(f"Unknown category '{category}'. Valid: {list(PLACING_SCORES_200m)}")
    return PLACING_SCORES_200m[cat].get(place, default)

# Filter to medal-placing rows only, then apply the rubric row by row
medal_rows_200m = Honors_200m[Honors_200m['Place'].isin([1, 2, 3])].copy()
medal_rows_200m['Placing_Score'] = medal_rows_200m.apply(lambda row: placing_score(row['Category'], row['Place']), axis=1)

# Sum scores per athlete
medal_score_200m = medal_rows_200m.groupby('Athlete')['Placing_Score'].sum().reset_index(name='medal_score')

# Merge into medals
medals_200m = medals_200m.merge(medal_score_200m, on='Athlete', how='left')
medals_200m['medal_score'] = medals_200m['medal_score'].fillna(0)


# ============================================================
# Average race category score (log & linear scales)
# ============================================================
log_scaled_score_map_200m = {
    'OW': 10.0, 'DF': 8.9, 'GW': 8.4, 'GL': 7.9, 'A': 7.4,
    'B': 6.4, 'C': 5.0, 'D': 3.8, 'E': 2.5, 'F': 1.0
}

linear_scaled_score_map_200m = {
    'OW': 10.0, 'DF': 7.0, 'GW': 6.0, 'GL': 5.2, 'A': 4.4,
    'B': 3.3, 'C': 2.2, 'D': 1.7, 'E': 1.3, 'F': 1.0
}

# Convert each race's category to its numeric score
clean_category_200m = Honors_200m['Category'].astype(str).str.upper().str.strip()
Honors_200m['category_score_log'] = clean_category_200m.map(log_scaled_score_map_200m)
Honors_200m['category_score_linear'] = clean_category_200m.map(linear_scaled_score_map_200m)

# Average across ALL of an athlete's races
avg_all_200m = Honors_200m.groupby('Athlete').agg(
    avg_category_log_200m=('category_score_log', 'mean'),
    avg_category_linear_200m=('category_score_linear', 'mean')
).reset_index()

# Average across only the races where the athlete placed 1, 2, or 3
avg_medal_200m = Honors_200m[Honors_200m['Place'].isin([1, 2, 3])].groupby('Athlete').agg(
    avg_medal_category_log_200m=('category_score_log', 'mean'),
    avg_medal_category_linear_200m=('category_score_linear', 'mean')
).reset_index()

medals_200m = medals_200m.merge(avg_all_200m, on='Athlete', how='left')
medals_200m = medals_200m.merge(avg_medal_200m, on='Athlete', how='left')


# ============================================================
# Average race category score (log & linear string value)
# ============================================================

#Method 1
#def nearest_category(value, score_map):
    #"""Return the category label whose score is closest to the given average."""
    #if pd.isna(value):
    #    return None
    #return min(score_map, key=lambda k: abs(score_map[k] - value))


#Method 2
# Threshold labels: F is <= 1.0, E is (1.0, 2.5], and so on (log scale example)
#def threshold_labels(series, score_map):
    #ordered = sorted(score_map.items(), key=lambda kv: kv[1])
    #labels = [k for k, _ in ordered]
    #bins = [-np.inf] + [v for _, v in ordered]
    #return pd.cut(series, bins=bins, labels=labels, right=True).astype(object)

#medals_200m['avg_category_log_str'] = threshold_labels(medals['avg_category_log'], log_scaled_score_map)
#medals_200m['avg_category_linear_str'] = threshold_labels(medals['avg_category_linear'], linear_scaled_score_map)

                      
# ============================================================
# Finalize as a pandas DataFrame
# ============================================================
medals_200m = pd.DataFrame(medals_200m)

# Export to CSV (uncomment when ready to save)
# medals.to_csv('medals.csv', index=False)

medals_200m

Let's add the sprinter's photo to the database by merging it with the photo csv

In [ ]:
sprinter_photo = pd.read_csv("../data/Sprinter_Photo.csv")

In [ ]:
sprinter_photo.head(3)

In [ ]:
#Helps with data cleaning
sprinter_photo['Athlete'] = sprinter_photo['Athlete'].str.strip()

In [ ]:
#Replace for data cleaning
sprinter_photo['Athlete'] = sprinter_photo['Athlete'].replace({'Christophe Lamaitre': 'Christophe Lemaitre'})

In [ ]:
sprinters = pd.merge(sprinters, sprinter_photo, on=['Athlete'],how='left')
sprinters.head(3)

Let's add second database to join number of season to each events dataframe (For Years Competed in each event).

In [ ]:
sprinters_df2 = pd.read_csv("../data/Worlds_Fastest_Sprinters_Master_List_Yearly_Progression.csv")
sprinters_df2.head(3)

In [ ]:
#Helps with data cleaning
sprinters_df2['Athlete'] = sprinters_df2['Athlete'].str.strip()

In [ ]:
#Replace for data cleaning
sprinters_df2['Athlete'] = sprinters_df2['Athlete'].replace({'Christophe Lamaitre': 'Christophe Lemaitre'})

In [ ]:
yrs_competed_200m = sprinters_df2[sprinters_df2['Event'] == '200m'].groupby('Athlete').size().reset_index(name='Years')

In [ ]:
yrs_competed_200m = yrs_competed_200m.sort_values(by='Years', ascending=False)
yrs_competed_200m.head()

In [ ]:
yrs_competed_200m = pd.DataFrame(yrs_competed_200m)
yrs_competed_200m.head(3)

Use this line of code to find a specific athlete in the data frame

In [ ]:
#yrs_competed_200m.loc[(yrs_competed_100m.Athlete == 'Frankie Fredericks')]

In [ ]:
#yrs_competed_200m.isnull()

In [ ]:
#yrs_competed_200m_nulls = yrs_competed_200m[yrs_competed_200m.isnull().any(axis=1)]
#print(yrs_competed_200m_nulls)

In [ ]:
yrs_competed_200m

In [ ]:
#Extract Excel file for data fix if needed

'''
xlwriter = pd.ExcelWriter('200M_Years_Competed.xlsx')
yrs_competed_200m.to_excel(xlwriter, sheet_name='Years Competed 200m')
xlwriter.close()
'''

Let's add a third database which incorportae's every race in each athletes's career.

In [ ]:
sprinters_df3 = pd.read_csv("../data/Sprinter_Career.csv")
sprinters_df3.head(3)

In [ ]:
#Helps with data cleaning
sprinters_df3['Athlete'] = sprinters_df3['Athlete'].str.strip()

In [ ]:
#Replace for data cleaning
sprinters_df3['Athlete'] = sprinters_df3['Athlete'].replace({'Christophe Lamaitre': 'Christophe Lemaitre'})

In [ ]:
All_200m_Races = sprinters_df3[sprinters_df3['Event'] == '200m']
All_200m_Races.head(3)

In [ ]:
#All_200m_Races['Place'] = pd.to_numeric(All_200m_Races['Place'])

#All_200m_Races['Place'] = All_200m_Races['Place'].astype('float64')  # or .astype(float)

All_200m_Races['Place'] = pd.to_numeric(All_200m_Races['Place'], errors='coerce')

Let's age the age of the athlete for each race completed

First we'll need to merge the Sprinter_DOB dataframe with the All_100m_Races dataframe

In [ ]:
All_200m_Races = pd.merge(All_200m_Races, Sprinter_DOB, on=['Athlete'],how='left')
All_200m_Races.head(3)

In [ ]:
# Convert to datetime
All_200m_Races['Date'] = pd.to_datetime(All_200m_Races['Date'], format='%d-%b-%y', errors='coerce')
All_200m_Races['DOB'] = pd.to_datetime(All_200m_Races['DOB'])

# Age in completed years at race date
All_200m_Races['Age_At_Race'] = (
    All_200m_Races['Date'].dt.year - All_200m_Races['DOB'].dt.year
    - (
        (All_200m_Races['Date'].dt.month < All_200m_Races['DOB'].dt.month)
        | (
            (All_200m_Races['Date'].dt.month == All_200m_Races['DOB'].dt.month)
            & (All_200m_Races['Date'].dt.day < All_200m_Races['DOB'].dt.day)
        )
    )
)

All_200m_Races

In [ ]:
#Let's drop the DOB column

#All_200m_Races.drop(columns=['DOB'], axis=1)

#All_200m_Races.head(3)

In [ ]:
All_200m_Races = All_200m_Races[['Athlete','Year','Event','Time','Date','Location','Legal','Meet_Type','Wind','Location_Country','Category','Race','Place','Score','Location_Country_FN','Location_Continent','Age_At_Race']]
All_200m_Races.head(3)

Let's drop all the races that were DNS, DNF, or DQ

In [ ]:
All_200m_Races = All_200m_Races[~All_200m_Races['Time'].isin(['DNS', 'DNF', 'DQ'])]
All_200m_Races.head(3)

Let's make sure that the time column is now a numeric datatype

In [ ]:
All_200m_Races['Time'] = pd.to_numeric(All_200m_Races['Time'], errors='coerce')

Let's drop all the Indoor marks

In [ ]:
All_200m_Races = All_200m_Races[All_200m_Races['Meet_Type'] != 'Indoor']
All_200m_Races.head(3)

Let's drop times that aren't legal from the dataframe

In [ ]:
All_200m_Races = All_200m_Races[All_200m_Races['Legal'] != 'NO']
All_200m_Races.head(3)

Let's Change any non-numeric values to either blank or 0 in the Wind column

To changes the non-numeric values values to blank

In [ ]:
All_200m_Races['Wind'] = All_200m_Races['Wind'].replace('NWI',np.nan)

To chabge the non-numeric valuesvalues to 0

In [ ]:
#All_200m_Races['Wind'] = All_200m_Races['Wind'].replace('NWI',0)

If I need to change the blank / null values in the wind column to 0

In [ ]:
#All_200m_Races['Wind'] = All_200m_Races['Wind'].fillna(0)

Let's make sure that the wind column is now a numeric datatype

In [ ]:
All_200m_Races['Wind'] = pd.to_numeric(All_200m_Races['Wind'], errors='coerce')

Let's drop all races that have Illegal wind (> 2.0)

In [ ]:
All_200m_Races = All_200m_Races[All_200m_Races['Wind'] <= 2.0]
All_200m_Races.head(3)

Let's Remove Any 200m Straight Races

In [ ]:
'''
Straight_Races_200m = All_200m_Races[
    All_200m_Races['Location'].str.contains(
    'Manchester Great Manchester Games, Manchester|Somerville adidas Boost Boston Games, Somerville, MA|Boston adidas Boost Boston Games, Boston, MA|adidas Boost Boston Games, Boston, MA|adidas Atlanta City Games, Piedmont Park, Atlanta, GA',
    case=False,
    na=False
    )
]

#I'll need to also drop Tommie Smith's 19.4 on May 7 1966 in San Jose

Straight_Races_200m

'''
# 200m Straight Competitors: Noah Lyles, Jereem Richards, Wallace Spearmon Jr, Steven Gardiner, Alonso Edward, Tommie Smith

In [ ]:
#TS = All_200m_Races.query('Athlete == "Tommie Smith" and Event == "200m"')
#TS

#TG = All_200m_Races.query('Athlete == "Tyson Gay" and Event == "200m"')
#TG

#WVN = All_200m_Races.query('Athlete == "Wayde van Niekerk" and Event == "200m"')
#WVN

In [ ]:
'''
tyson_200 = All_200m_Races[(All_200m_Races["Athlete"] == "Tyson Gay") & (All_200m_Races["Event"] == "200m")]
fastest = tyson_200["Time"].min()
print(fastest)
'''

In [ ]:
'''
search_row = All_200m_Races[
    (All_200m_Races["Athlete"] == "Tyson Gay")
    & (All_200m_Races["Event"] == "200m")
    & (All_200m_Races["Time"] == 19.41)
]
print(search_row)
'''

In [ ]:
#Straight_Races_200m = All_200m_Races.loc[[1690, 2599, 3666, 4684, 5913]] #Insert later if needed 1885,

#Old index numbers
#All_200m_Races.loc[[4895, 5030, 5107, 7378, 8951, 11953, 12638, 14406, 18165]]


#Straight_Races_200m

In [ ]:
#len(Straight_Races_200m)

Let's drop those races from the main dataframe

In [ ]:
#All_200m_Races = All_200m_Races.drop([4895, 5030, 5107, 7378, 8951, 11953, 12638, 14406, 18165])

#All_200m_Races = All_200m_Races.drop([5030, 7378, 11953, 12638, 14406])

Let's get the Caeer Average for Each Athlete

In [ ]:
Career_average_200m = All_200m_Races.groupby('Athlete')['Time'].mean().reset_index(name='Career_Avg_200m')

In [ ]:
Career_average_200m = pd.DataFrame(Career_average_200m)
Career_average_200m.head(3)

In [ ]:
#All_200m_Races = pd.merge(All_200m_Races, Career_average_200m, on=['Athlete'],how='left')
#All_200m_Races.head(3)

Years Competed For Each Athlete (Alternative Option)

In [ ]:
#YRS_Competed = All_200m_Races.groupby('Athlete')['Year'].nunique().reset_index(name='Years_Competed')
#YR_Competed = YR_Com.sort_values('Years_Competed', ascending=False)

#YR_Competed

Seasons Best For Each Athlete (Alternative Option)

In [ ]:
Season_Bests_200m = All_200m_Races.groupby(["Athlete", "Year"])["Time"].min()

Avg_Season_Best_200m = (Season_Bests_200m.groupby("Athlete").mean().reset_index().rename(columns={"Time": "Avg_Season_Best_200m"}))

Avg_Season_Best_200m

In [ ]:
Avg_Season_Best_200m = pd.DataFrame(Avg_Season_Best_200m)
Avg_Season_Best_200m.head(3)

Total Races for Each Athlete

In [ ]:
athlete_race_count_200m = All_200m_Races.groupby('Athlete').size().reset_index(name='total_races_200m')
athlete_race_count_200m.head(3)

In [ ]:
athlete_race_count_200m = pd.DataFrame(athlete_race_count_200m)
athlete_race_count_200m.head(3)

In [ ]:
athlete_race_count_200m

Number of races for each athlete by year

In [ ]:
#athlete_race_count_per_year_200m = hundred_meter_races.groupby(['Athlete', 'Year']).size().reset_index(name='races_per_year_200m')
#athlete_race_count_per_year_200m.head(3)

Sub-20 200m For each athlete

In [ ]:
sub_20_200m = pd.DataFrame((All_200m_Races['Time'] < 20).groupby(All_200m_Races['Athlete']).sum().sort_values(ascending=False))

sub_20_200m

Average Wind For Each Athlete

In [ ]:
Average_Wind_200m = All_200m_Races.groupby('Athlete')['Wind'].mean().reset_index(name='Avg_Wind_200m')
Average_Wind_200m.head(3)

In [ ]:
Average_Wind_200m = pd.DataFrame(Average_Wind_200m)
Average_Wind_200m.head(3)

Total Wins Per Athlete

In [ ]:
Wins_200m = All_200m_Races[All_200m_Races['Place'] == 1].groupby('Athlete').size().reset_index(name='Wins_200m')
Wins_200m

In [ ]:
Wins_200m = pd.DataFrame(Wins_200m)
Wins_200m.head(3)

Average Place (Finish) For Each Athlete

In [ ]:
Average_Place_200m = All_200m_Races.groupby('Athlete')['Place'].mean().reset_index(name='Avg_Place_200m')
Average_Place_200m.head(3)

Let's Find the Average Place when the Race is a Final

In [ ]:
Final_Heat_200m = All_200m_Races[All_200m_Races['Race'] =='F']
Final_Heat_200m.head(3)

In [ ]:
Average_Place_200mF = Final_Heat_200m.groupby('Athlete')['Place'].mean().reset_index(name='Avg_Place_200m_Final_Heat')
Average_Place_200mF.head(3)

In [ ]:
Average_Place_200mF = pd.DataFrame(Average_Place_200mF)
Average_Place_200mF.head(3)

Let's Merge this with the Average 200m Place Dataframe

In [ ]:
Average_Place_200m = pd.merge(Average_Place_200m, Average_Place_200mF, on=['Athlete'],how='left')
Average_Place_200m.head(3)

Let's Find the Average Place when the Race is a Final in a Championship or High Level Meet

In [ ]:
High_Level_Meet = ['OW','DF', 'DL', 'GW', 'GL']

In [ ]:
High_Level_Meet_Final = All_200m_Races[(All_200m_Races['Race'] =='F') & (All_200m_Races['Category'].isin(High_Level_Meet))]
High_Level_Meet_Final.head(3)

In [ ]:
High_Level_Meet_Final = pd.DataFrame(High_Level_Meet_Final)
High_Level_Meet_Final.head(3)

In [ ]:
Avg_Clutch_Finish_200mF = High_Level_Meet_Final.groupby('Athlete')['Place'].mean().reset_index(name='Avg_Clutch_Finish_200mF')
Avg_Clutch_Finish_200mF.head(3)

Let's Merge this with the Average 200m Place Dataframe as well

In [ ]:
Average_Place_200m = pd.merge(Average_Place_200m, Avg_Clutch_Finish_200mF, on=['Athlete'],how='left')
Average_Place_200m.head(3)

In [ ]:
Average_Place_200m = pd.DataFrame(Average_Place_200m)
Average_Place_200m.head(3)

Total World Athletics Score

In [ ]:
Total_WA_Score_200m = All_200m_Races.groupby('Athlete')['Score'].sum().reset_index(name='Total_WA_Score_200m')
Total_WA_Score_200m.head(3)

In [ ]:
Total_WA_Score_200m = pd.DataFrame(Total_WA_Score_200m)
Total_WA_Score_200m.head(3)

Average World Athletics Score

In [ ]:
Avg_WA_Score_200m = All_200m_Races.groupby('Athlete')['Score'].mean().reset_index(name='Avg_WA_Score_200m')
Avg_WA_Score_200m.head(3)

In [ ]:
Avg_WA_Score_200m = pd.DataFrame(Avg_WA_Score_200m)
Avg_WA_Score_200m.head(3)

## World Athletics Category Rankings (Move New Data Up Here)

| Place | OW  | DF  | GW  | GL  | A   | B   | C   | D   | E   | F   |
|-------|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|
| 1st   | 350 | 240 | 200 | 170 | 140 | 100 | 60  | 40  | 25  | 15  |
| 2nd   | 310 | 210 | 170 | 145 | 120 | 80  | 50  | 35  | 21  | 10  |
| 3rd   | 280 | 185 | 150 | 130 | 110 | 70  | 45  | 30  | 18  | 5   |
| 4th   | 250 | 170 | 140 | 120 | 100 | 60  | 40  | 25  | 15  |     |
| 5th   | 230 | 155 | 130 | 110 | 90  | 55  | 35  | 22  | 12  |     |
| 6th   | 215 | 145 | 120 | 100 | 80  | 50  | 30  | 19  | 10  |     |
| 7th   | 200 | 135 | 110 | 90  | 70  | 45  | 27  | 17  |     |     |
| 8th   | 185 | 125 | 100 | 80  | 60  | 40  | 25  | 15  |     |     |
| 9th   | 130 | 90  | 70  | 60  |     |     |     |     |     |     |
| 10th  | 120 | 80  | 60  | 50  |     |     |     |     |     |     |
| 11th  | 110 | 70  | 50  | 45  |     |     |     |     |     |     |
| 12th  | 100 | 60  | 45  | 40  |     |     |     |     |     |     |
| 13th  | 95  |     |     |     |     |     |     |     |     |     |
| 14th  | 90  |     |     |     |     |     |     |     |     |     |
| 15th  | 85  |     |     |     |     |     |     |     |     |     |
| 16th  | 80  |     |     |     |     |     |     |     |     |     |

*Source: [World Athletics – Basics of the World Rankings](https://worldathletics.org/world-ranking-rules/basics)*

## Category Scale: Log-Transformed, Rescaled to 1–10

Using the raw 1st-place Placing Scores directly isn't ideal for averaging — they span **15 to 350, roughly a 23x range**, which is too wide and skewed to treat as a linear scale. A pure ordinal scale (OW=10, DF=9, GW=8...) fixes the range problem but assumes every category is equally far apart, which isn't true either — the gap between OW and DF is much bigger than the gap between D and E.

**The approach:** take the log of each category's 1st-place point value, then linearly rescale that log range onto a clean 1–10 scale (OW=10, F=1). This compresses the extreme top-end gap (OW/DF) while still preserving that higher tiers are worth meaningfully more, and it keeps mid/low tiers (B, C, D) from collapsing too close to 1.

| Category | 1st-place points | Scaled score (1–10) |
|----------|------------------|----------------------|
| OW       | 350              | 10.0                 |
| DF       | 240              | 8.9                  |
| GW       | 200              | 8.4                  |
| GL       | 170              | 7.9                  |
| A        | 140              | 7.4                  |
| B        | 100              | 6.4                  |
| C        | 60               | 5.0                  |
| D        | 40               | 3.8                  |
| E        | 25               | 2.5                  |
| F        | 15               | 1.0                  |

**Trade-off to keep in mind:** this scale spreads scores across the *full* 1–10 range fairly evenly, so a runner who mostly races Category B/C still lands mid-scale rather than near the bottom. If you instead want the scale to sharply reward *only* top-tier competition (and treat everything below GL as roughly similar), a linear rescale of the raw points — without the log step — would compress B through F much closer to 1.

In [ ]:
#Scaled Score Mapping dictionary
log_scaled_score_map = {
    'OW': 10.0,
    'DF': 8.9,
    'GW': 8.4,
    'GL': 7.9,
    'A': 7.4,
    'B': 6.4,
    'C': 5.0,
    'D': 3.8,
    'E': 2.5,
    'F': 1.0
}

# Create new column
All_200m_Races['Categorey_Log_Scaled_Score'] = All_200m_Races['Category'].map(log_scaled_score_map)

In [ ]:
Avg_Race_Cateogry_Score_Log = All_200m_Races.groupby('Athlete')['Categorey_Log_Scaled_Score'].mean().reset_index(name='Avg_Log_Cat_Score_200m')
Avg_Race_Cateogry_Score_Log.head(3)

In [ ]:
Avg_Race_Cateogry_Score_Log = pd.DataFrame(Avg_Race_Cateogry_Score_Log)
Avg_Race_Cateogry_Score_Log.head(3)

## Category Scale: Pure Linear Rescale to 1–10

This version rescales the raw 1st-place Placing Scores directly onto a 1–10 range (OW=10, F=1), without the log-compression step. Because it preserves the full ~23x ratio between OW (350) and F (15), it produces a much more top-heavy scale: GL through F cluster in the bottom third, while OW and DF remain far apart at the top.

| Category | 1st-place points | Scaled score (1–10, linear) |
|----------|------------------|------------------------------|
| OW       | 350              | 10.0                         |
| DF       | 240              | 7.0                          |
| GW       | 200              | 6.0                          |
| GL       | 170              | 5.2                          |
| A        | 140              | 4.4                          |
| B        | 100              | 3.3                          |
| C        | 60               | 2.2                          |
| D        | 40               | 1.7                          |
| E        | 25               | 1.3                          |
| F        | 15               | 1.0                          |

**Use this version if:** you want the ranking system to sharply reward top-tier competition (OW/DF/GW) and treat mid-to-lower tiers as roughly comparable to each other.

In [ ]:
#Scaled Score Mapping dictionary
linear_scaled_score_map = {
    'OW': 10.0,
    'DF': 7.0,
    'GW': 6.0,
    'GL': 5.2,
    'A': 4.4,
    'B': 3.3,
    'C': 2.2,
    'D': 1.7,
    'E': 1.3,
    'F': 1.0
}

# Create new column
All_200m_Races['Categorey_Linear_Scaled_Score'] = All_200m_Races['Category'].map(linear_scaled_score_map)

In [ ]:
Avg_Race_Cateogry_Score_Lin = All_200m_Races.groupby('Athlete')['Categorey_Linear_Scaled_Score'].mean().reset_index(name='Avg_Lin_Cat_Score_200m')
Avg_Race_Cateogry_Score_Lin.head(3)

In [ ]:
Avg_Race_Cateogry_Score_Lin = pd.DataFrame(Avg_Race_Cateogry_Score_Lin)
Avg_Race_Cateogry_Score_Lin.head(3)

In [ ]:
athlete_category_summary = (
    All_200m_Races.groupby(['Athlete', 'Category'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

athlete_category_summary['win_pct'] = (
    athlete_category_summary['wins']
    / athlete_category_summary['total_races']
    * 100
).round(1)

athlete_category_summary.drop(columns='wins', inplace=True)

In [ ]:
athlete_category_summary_200m = pd.DataFrame(athlete_category_summary)
athlete_category_summary_200m.head(3)

If we want to pivot this table to 1 row per athlete

In [ ]:
pivot = athlete_category_summary.pivot(
    index='Athlete',
    columns='Category',
    values='win_pct'
)

Year By Year Comparision

In [ ]:
athlete_category_year_summary = (
    All_200m_Races.groupby(['Athlete', 'Year', 'Category'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

athlete_category_year_summary['win_pct'] = (
    athlete_category_year_summary['wins']
    / athlete_category_year_summary['total_races']
    * 100
).round(1)

athlete_category_year_summary.drop(columns='wins', inplace=True)

In [ ]:
athlete_category_year_summary_200m = pd.DataFrame(athlete_category_year_summary)
athlete_category_year_summary_200m.head(3)

Pivot Table Version of This

In [ ]:
pivot = athlete_category_year_summary.pivot_table(
index=['Athlete', 'Category'],
columns='Year',
values='win_pct'
)

Athlete's overall stats each year regardless of category

In [ ]:
athlete_year_summary = (
    All_200m_Races.groupby(['Athlete', 'Year'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

athlete_year_summary['win_pct'] = (
    athlete_year_summary['wins']
    / athlete_year_summary['total_races']
    * 100
).round(1)


In [ ]:
athlete_year_summary_200m = pd.DataFrame(athlete_year_summary)
athlete_year_summary_200m.head(3)

Let's Analyze Each Athletes Race Stats By Country

In [ ]:
country_category_summary = (
    All_200m_Races.groupby(['Athlete', 'Location_Country_FN', 'Category'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

country_category_summary['win_pct'] = (
    country_category_summary['wins']
    / country_category_summary['total_races']
    * 100
).round(1)

country_category_summary.drop(columns='wins', inplace=True)

In [ ]:
country_category_summary_200m = pd.DataFrame(country_category_summary)
country_category_summary_200m.head(3)

Year By Year Analysis (Country)

In [ ]:
country_year_summary = (
    All_200m_Races.groupby(['Athlete', 'Location_Country_FN', 'Year'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

country_year_summary['win_pct'] = (
    country_year_summary['wins']
    / country_year_summary['total_races']
    * 100
).round(1)

country_year_summary.drop(columns='wins', inplace=True)

In [ ]:
country_year_summary_200m = pd.DataFrame(country_year_summary)
country_year_summary_200m.head(3)

Let's Analyze Each Athletes Race Stats By Continent

In [ ]:
continent_category_summary = (
    All_200m_Races.groupby(['Athlete', 'Location_Continent', 'Category'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

continent_category_summary['win_pct'] = (
    continent_category_summary['wins']
    / continent_category_summary['total_races']
    * 100
).round(1)

continent_category_summary.drop(columns='wins', inplace=True)

In [ ]:
continent_category_summary_200m = pd.DataFrame(continent_category_summary)
continent_category_summary_200m.head(3)

Year By Year Analysis (Continent)

In [ ]:
continent_year_summary = (
    All_200m_Races.groupby(['Athlete', 'Location_Continent', 'Year'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

continent_year_summary['win_pct'] = (
    continent_year_summary['wins']
    / continent_year_summary['total_races']
    * 100
).round(1)

continent_year_summary.drop(columns='wins', inplace=True)

In [ ]:
continent_year_summary_200m = pd.DataFrame(continent_year_summary)
continent_year_summary_200m.head(3)

Compact Version (Country & Continent)

In [ ]:
athlete_country_year_category = (
    All_200m_Races.groupby(
        ['Athlete', 'Location_Country_FN', 'Location_Continent', 'Year', 'Category']
    )
    .agg(
        total_races=('Event', 'count'),
        avg_wind=('Wind', 'mean'),
        avg_place=('Place', 'mean'),
        avg_score=('Score', 'mean'),
        wins=('Place', lambda x: (x == 1).sum())
    )
    .reset_index()
)

athlete_country_year_category['win_pct'] = (
    athlete_country_year_category['wins']
    / athlete_country_year_category['total_races']
    * 100
).round(1)

athlete_country_year_category.drop(columns='wins', inplace=True)

In [ ]:
athlete_country_year_category_200m = pd.DataFrame(athlete_country_year_category)
athlete_country_year_category_200m.head(3)

Meerge Data Season and total races data next

In [ ]:
seasons_and_races_200m = pd.merge(athlete_race_count_200m, yrs_competed_200m, on=['Athlete'],how='left')
seasons_and_races_200m.head(3)

Let's add the Sub-20 200m

In [ ]:
seasons_and_races_200m = pd.merge(seasons_and_races_200m, sub_20_200m, on=['Athlete'],how='left')
seasons_and_races_200m.head(3)

Let's add the career average for each athlete

In [ ]:
seasons_and_races_200m = pd.merge(seasons_and_races_200m, Career_average_200m, on=['Athlete'],how='left')
seasons_and_races_200m.head(3)

In [ ]:
seasons_and_races_200m['Avg_Races_Year_200m'] = round(seasons_and_races_200m['total_races_200m'] / seasons_and_races_200m['Years'],2)
sprinters.head()

Let's add the season best average (alternative version) for each athlete

In [ ]:
seasons_and_races_200m = pd.merge(seasons_and_races_200m, Avg_Season_Best_200m, on=['Athlete'],how='left')
seasons_and_races_200m.head(3)

Let's add the average wind for each athlete

In [ ]:
seasons_and_races_200m = pd.merge(seasons_and_races_200m, Average_Wind_200m, on=['Athlete'],how='left')
seasons_and_races_200m.head(3)

Let's add total wins for each athlete

In [ ]:
seasons_and_races_200m = pd.merge(seasons_and_races_200m, Wins_200m, on=['Athlete'],how='left')
seasons_and_races_200m.head(3)

Let's add average place (finish) for each athlete

In [ ]:
seasons_and_races_200m = pd.merge(seasons_and_races_200m, Average_Place_200m, on=['Athlete'],how='left')
seasons_and_races_200m.head(3)

Let's add the total World Athletics Score for Each Athlete

In [ ]:
seasons_and_races_200m = pd.merge(seasons_and_races_200m, Total_WA_Score_200m, on=['Athlete'],how='left')
seasons_and_races_200m.head(3)

Let's add the average World Athletics Score for Each Athlete

In [ ]:
seasons_and_races_200m = pd.merge(seasons_and_races_200m, Avg_WA_Score_200m, on=['Athlete'],how='left')
seasons_and_races_200m.head(3)

Let's add the 200m Medals data

In [ ]:
seasons_and_races_200m = pd.merge(seasons_and_races_200m, medals_200m, on=['Athlete'],how='left')
seasons_and_races_200m.head(3)

Let's add the average race cateory for each athlete's Log-Transformed Score

In [ ]:
seasons_and_races_200m = pd.merge(seasons_and_races_200m, Avg_Race_Cateogry_Score_Log, on=['Athlete'],how='left')
seasons_and_races_200m.head(3)

Let's add the average race cateory for each athlete's Linear Score

In [ ]:
seasons_and_races_200m = pd.merge(seasons_and_races_200m, Avg_Race_Cateogry_Score_Lin, on=['Athlete'],how='left')
seasons_and_races_200m.head(3)

In [ ]:
seasons_and_races_200m = pd.DataFrame(seasons_and_races_200m)
seasons_and_races_200m.head(3)

In [ ]:
seasons_and_races_200m

In [ ]:
#extract csv of seasons_and_races_100m dataframe to find all of the null values

#seasons_and_races_200m.to_csv("seasons_&_races_200m.csv", index=False)

In [ ]:
#Extract Excel file for data fix if needed

'''
xlwriter = pd.ExcelWriter('200M_Races_Seasons.xlsx')
seasons_and_races_200m.to_excel(xlwriter, sheet_name='Seasons & Races 200m')
xlwriter.close()
'''

Fixing Null Data

In [ ]:
#Old Batch
'''
seasons_and_races_200m.at[17, "Years"] = 17
seasons_and_races_200m.at[17, "Avg_Races_Year_200m"] = 9.71
seasons_and_races_200m.at[22, "Years"] = 7
seasons_and_races_200m.at[22, "Avg_Races_Year_200m"] = 4
seasons_and_races_200m.at[42, "Years"] = 7
seasons_and_races_200m.at[42, "Avg_Races_Year_200m"] = 2.14
seasons_and_races_200m.at[46, "Years"] = 6
seasons_and_races_200m.at[46, "Avg_Races_Year_200m"] = 2
seasons_and_races_200m.at[53, "Years"] = 10
seasons_and_races_200m.at[53, "Avg_Races_Year_200m"] = 4.4
seasons_and_races_200m.at[62, "Years"] = 10
seasons_and_races_200m.at[62, "Avg_Races_Year_200m"] = 5.7
seasons_and_races_200m.at[66, "Years"] = 13
seasons_and_races_200m.at[66, "Avg_Races_Year_200m"] = 5.23
'''

seasons_and_races_200m.at[1, "avg_category_log_200m"] = 7.9 #Abdul-Rasheed Saminu
seasons_and_races_200m.at[1, "avg_category_linear_200m"] = 5.2   #Abdul-Rasheed Saminu
seasons_and_races_200m.at[, "Total_Senior_Medals"] =     #
seasons_and_races_200m.at[, "Total_Diamond_League_Medals"] =    #
seasons_and_races_200m.at[, "Diamond_League_gold"] =    #
seasons_and_races_200m.at[, "Diamond_League_Medalist"] = 'TRUE'   #
seasons_and_races_200m.at[, "Diamond_League_Champion"] = 'TRUE'   #



In [ ]:
seasons_and_races_200m.loc[(seasons_and_races_200m.Athlete == 'Abdul-Rasheed Saminu')]

In [ ]:
#Let's first fix the data for the athletes where all medal columns should be zero or false


# Columns that came from medals (excluding the join key and the race-category column)
medals_200_cols = [c for c in medals_200m.columns if c not in ('Athlete', 'Category')]

# Split by dtype in the ORIGINAL medals dataframe
bool_cols = [c for c in medals_200_cols if medals_200m[c].dtype == bool]
num_cols = [c for c in medals_200_cols if c not in bool_cols]

# Pick the athletes, by name (or use the index version below)
athletes = ['Abdalelah Haroun','Ackeem Blake','Alvin Harrison','Angelo Taylor','Anthony Zambrano','Champion Allison',
            'Chris Bailey','Christopher Morales Williams','Clarence Munyai', 'Danny Everett', 'Erriyon Knighton', 
            'Gout Gout','Jacory Patterson', 'Jaiden Reid', 'James Dadzie','Jereem Richards','Jerome Young', 'José Figueroa',
            'Justin Braun', 'Luguelín Santos', 'Luxolo Adams', 'Matthew Hudson-Smith','Micah Williams', 'Michael Johnson',
            'Michael Norman','Nathon Allen','Quincy Watts','Rai Benjamin','Randolph Ross','Robert Gregory','Rusheen McDonald',
            'Sinesipho Dambile', 'Steve Lewis','Steven Gardiner', 'Tarsis Gracious Orogot','Udodi Chudi Onwuzurike',
            'Wallace Spearmon Jr.', 'Gary Kikaya','LaShawn Merritt','Warren Weir','Wayde van Niekerk', 'Zakithi Nene']  # replace with names

mask = seasons_and_races_200m['Athlete'].isin(athletes)


# Index labels from the JOINED dataframe
#idx = [3, 5, 7, 8, 11, 14, 17, 20, 22, 24, 28, 34, 36, 38, 39, 40, 41, 42, 46, 47, 53, 59, 60, 62, 64, 66, 68, 70, 77, 78, 80, 84, 87, 89, 91, 92, 94, 97, 102, 104, 106, 107 110] # replace with your index numbers
idx = [89, 96]

# Keep only labels that actually exist in the joined dataframe
idx = [i for i in idx if i in seasons_and_races_100m.index]

# Or, if you'd rather use index labels from the joined dataframe:
mask2 = seasons_and_races_100m.index.isin(idx)


# Apply the changes
seasons_and_races_200m.loc[mask, num_cols] = 0
seasons_and_races_200m.loc[mask, bool_cols] = False
seasons_and_races_200m.loc[mask2, num_cols] = 0
seasons_and_races_200m.loc[mask2, bool_cols] = False

In [ ]:
# Columns that came from medals (excluding the join key and the race-category column)
medals_200_cols = [c for c in medals_200m.columns if c not in ('Athlete', 'Category')]

# Split by dtype in the ORIGINAL medals dataframe
bool_cols = [c for c in medals_200_cols if medals_200m[c].dtype == bool]
num_cols = [c for c in medals_200_cols if c not in bool_cols]

# Pick the athletes, by name (or use the index version below)
athletes = ['Abdul-Rasheed Saminu','']  # replace with names #Possibly add Joe DeLoach and Christophe Lemaitre if previous code doesn't work
mask = seasons_and_races_200m['Athlete'].isin(athletes)

# Index labels from the JOINED dataframe
#idx = [1,]  # replace with your index numbers
#idx = []  # replace with your index numbers


# Keep only labels that actually exist in the joined dataframe
#idx = [i for i in idx if i in seasons_and_races_100m.index]

# Or, if you'd rather use index labels from the joined dataframe:
#mask2 = seasons_and_races_200m.index.isin(idx)

# Fill only the blanks for those athletes
seasons_and_races_200m.loc[mask, num_cols] = seasons_and_races_200m.loc[mask, num_cols].fillna(0)
seasons_and_races_200m.loc[mask, bool_cols] = seasons_and_races_200m.loc[mask, bool_cols].fillna(False)
#seasons_and_races_200m.loc[mask2, num_cols] = seasons_and_races_200m.loc[mask, num_cols].fillna(0)
#seasons_and_races_200m.loc[mask2, bool_cols] = seasons_and_races_200m.loc[mask, bool_cols].fillna(False)


In [ ]:
seasons_and_races_200m

In [ ]:
#seasons_and_races_200m.loc[(seasons_and_races_200m.Athlete == 'Oblique Seville')]

In [ ]:
#seasons_and_races_200m.loc[(seasons_and_races_200m.Athlete == 'Frankie Fredericks')]

In [ ]:
#seasons_and_races_200m.loc[(seasons_and_races_200m.Athlete == 'Pietro Mennea')]

In [ ]:
#seasons_and_races_200m.loc[(seasons_and_races_200m.Athlete == 'Joe DeLoach')]

In [ ]:
#seasons_and_races_200m.loc[(seasons_and_races_200m.Athlete == 'Wayde van Niekerk')]

In [ ]:
# We don't need the 'Avg_Season_Best_200m' from the df_200m now that we have it coming from the 'seasons_and_races' df
df_200m= sprinters[['Athlete', 'Country','Continent','Status', 'DOB','Year Born','Month Born','Decade Born','200_PR','T25_200_All_Time_Rank','T25_200_AT_RK_NUM']]
df_200m

In [ ]:
df_200m['SB_Avg_200m_PR_Diff'] = round(df_200m['Avg_Season_Best_200m'] - df_200m['200_PR'],2)
df_200m.head(3)

In [ ]:
df_200m = pd.merge(df_200m, seasons_and_races_200m, on=['Athlete'],how='left')
df_200m.head(3)

In [ ]:
df_200m = df_200m.rename(columns={'Years': 'Years_Competed_200m'})
df_200m.head(3)

In [ ]:
df_200m['Career_Avg_200m_PR_Diff'] = round(df_200m['Career_Avg_200m'] - df_200m['200_PR'],2)
df_200m.head(3)

In [ ]:
df_200m['Win_Pct_200m'] = round((df_200m['Wins_200m'] / df_200m['total_races_200m'])*100,2)
df_200m.head(3)

In [ ]:
#df_200m = df_200m.sort_values(by='Avg_Season_Best_200m', ascending=False)
#df_200m

In [ ]:
#df_200m = df_200m.sort_values(by='Avg_Season_Best_200m')
#df_200m

In [ ]:
df_200m = df_200m.dropna(subset='200_PR')
df_200m

In [ ]:
#Extract Excel file for data fix if needed


'''
xlwriter = pd.ExcelWriter('200M.xlsx')
df_200m.to_excel(xlwriter, sheet_name='200m')
xlwriter.close()
'''

Fixing Null Values

In [ ]:
df_200m.at[12, "total_races_200m"] = 15
df_200m.at[12, "Years_Competed_200m"] = 7
df_200m.at[12, "Career_Avg_200m"] = 21.659333
df_200m.at[12, "Avg_Races_Year_200m"] = 2.14
df_200m.at[12, "Career_Avg_200m_PR_Diff"] = .80
df_200m.at[28, "total_races_200m"] = 165
df_200m.at[28, "Years_Competed_200m"] = 17
df_200m.at[28, "Career_Avg_200m"] = 20.311576
df_200m.at[28, "Avg_Races_Year_200m"] = 9.71
df_200m.at[28, "Career_Avg_200m_PR_Diff"] = .63
df_200m.at[39, "total_races_200m"] = 137
df_200m.at[39, "Years_Competed_200m"] = 16
df_200m.at[39, "Career_Avg_200m"] =20.48708
df_200m.at[39, "Avg_Races_Year_200m"] = 8.56
df_200m.at[39, "Career_Avg_200m_PR_Diff"] = .77
df_200m.at[42, "total_races_200m"] = 28
df_200m.at[42, "Years_Competed_200m"] = 7
df_200m.at[42, "Career_Avg_200m"] = 20.496786
df_200m.at[42, "Avg_Races_Year_200m"] = 4.0
df_200m.at[42, "Career_Avg_200m_PR_Diff"] = .75
df_200m.at[44, "total_races_200m"] = 57
df_200m.at[44, "Years_Competed_200m"] = 10
df_200m.at[44, "Career_Avg_200m"] = 20.695965
df_200m.at[44, "Avg_Races_Year_200m"] = 5.7
df_200m.at[44, "Career_Avg_200m_PR_Diff"] = .86

In [ ]:
df_200m

In [ ]:
df_200m = df_200m.sort_values(by='Career_Avg_200m')
df_200m

In [ ]:
df_200m = pd.DataFrame(df_200m)

# ***Statistical Analysis 200m***

In [ ]:
from scipy.stats import zscore
from sklearn.linear_model import LinearRegression

Linear Regression

In [ ]:
# Function to calculate standard deviation (consistency)
df_200m['consistency'] = df_200m['Avg_Season_Best_200m']  # Placeholder: You could replace with actual std per year data if available

### 1. Regression Analysis ###
# Linear regression: relationship between average_time and years_competed
X = df_200m[['Years_Competed_200m','200_PR']]
y = df_200m['Avg_Season_Best_200m']

# Fit model
model = LinearRegression()
model.fit(X, y)

# Predictions and residuals
df_200m['predicted_time'] = model.predict(X)
df_200m['residuals'] = df_200m['Avg_Season_Best_200m'] - df_200m['predicted_time']

print("Regression coefficients (slope):", model.coef_)
print("Intercept:", model.intercept_)


Z-Score Standardization

Version 1

In [ ]:
'''
# Z-score for average_time and years_competed
df_200m['z_time'] = zscore(df_200m['Avg_Season_Best_200m'])
df_200m['z_years'] = zscore(df_200m['Years_Competed_200m'])
df_200m['z_PR'] = zscore(df_200m['200_PR'])
df_200m['z_career_avg_season'] = zscore(df_200m['Career_Avg_200m'])
df_200m['z_total_races'] = zscore(df_200m['total_races_200m'])
df_200m['z_races_per_year'] = zscore(df_200m['Avg_Races_Year_200m'])

# Z-score comparison (combine time and years)
df_200m['z_combined'] = (df_200m['z_time'] + df_200m['z_years'] + df_200m['z_PR']+ df_200m['z_career_avg_season'] + df_200m['z_total_races'] + df_200m['z_races_per_year']) / 6

'''

Version 2 (Takes into the account the that the total races and avg races per year has a major affect on the combined z-score)

In [ ]:
df_200m['z_time'] = zscore(df_200m['Avg_Season_Best_200m'],nan_policy='omit')
df_200m['z_years'] = -zscore(df_200m['Years_Competed_200m'],nan_policy='omit')
df_200m['z_PR'] = zscore(df_200m['200_PR'],nan_policy='omit')
df_200m['z_career_avg_season'] = zscore(df_200m['Career_Avg_200m'],nan_policy='omit')
df_200m['z_total_races'] = -zscore(df_200m['total_races_200m'],nan_policy='omit')
df_200m['z_races_per_year'] = -zscore(df_200m['Avg_Races_Year_200m'],nan_policy='omit')
#Need to determine if the following below need a positive or negative z-socre
#df_200m['z_avg_wind_per_year'] = -zscore(df_200m['Average_Wind_200m'],nan_policy='omit')
#df_200m['z_avg_place_per_race'] = -zscore(df_200m['Average_Place_200m'],nan_policy='omit')
#df_200m['z_avg_place_per_final'] = -zscore(df_200m['Avg_Place_200m_Final_Heat'],nan_policy='omit')
#df_200m['z_avg_clutch_finish'] = -zscore(df_200m['Avg_Clutch_Finish_200mF'],nan_policy='omit')
#df_200m['z_total_WA_score'] = -zscore(df_200m['Total_WA_Score_200m'],nan_policy='omit')
#df_200m['z_avg_WA_score'] = -zscore(df_200m['Avg_WA_Score_200m'],nan_policy='omit')

# Z-score comparison (combine time and years)
df_200m['z_combined'] = (df_200m['z_time'] + df_200m['z_years'] + df_200m['z_PR']+ df_200m['z_career_avg_season'] + df_200m['z_total_races'] + df_200m['z_races_per_year']) / 6

Alternate Ranking

In [ ]:
df_200m['alt_rank'] = df_200m['z_combined'].rank(ascending=True)

In [ ]:
df_200m.sort_values('alt_rank').head(10)

In [ ]:
Z_Score_Ranking_200m = df_200m.sort_values('alt_rank')

In [ ]:
Z_Score_Ranking_200m = pd.DataFrame(Z_Score_Ranking_200m)
Z_Score_Ranking_200m.head(3)

In [ ]:
#Let's make the entire dataframe a figure factory table


fig =  ff.create_table(Z_Score_Ranking_200m)
fig.show()

#fig.write_html("Z_Score_Ranking_200m.html")
#fig.write_image("Z_Score_Ranking_200m.svg")

Efficiency / Ratio Analysis

In [ ]:
# Efficiency score (average_time per year competed)
df_200m['efficiency_score'] = df_200m['Avg_Season_Best_200m'] / df_200m['Years_Competed_200m']

# Efficiency score: How close the sprinter's average season best to their personal best
df_200m['efficiency_score_sb'] = df_200m['200_PR'] / df_200m['Avg_Season_Best_200m']

# Efficiency score: How close the sprinter's average is to their personal best
df_200m['efficiency_score_pr'] = df_200m['200_PR'] / df_200m['Career_Avg_200m']

Ranking System

In [ ]:
# Combine rankings based on average_time, consistency, and longevity (years_competed)
df_200m['rank_personal_best'] = df_200m['200_PR'].rank(ascending=True)  # Lower personal best is better
df_200m['rank_average_sb'] = df_200m['Avg_Season_Best_200m'].rank(ascending=True)  # Lower is better
df_200m['rank_career_avg'] = df_200m['Career_Avg_200m'].rank(ascending=True)
#df_200m['rank_consistency'] = df_200m['consistency'].rank(ascending=True)  # Lower std dev is better
df_200m['rank_consistency'] = df_200m['consistency'].abs().rank(ascending=True) # Lower residuals (consistency) is better
df_200m['rank_years_competed'] = df_200m['Years_Competed_200m'].rank(ascending=False)  # Longer careers are better
df_200m['rank_total_races'] = df_200m['total_races_200m'].rank(ascending=False)  # More races is better
df_200m['rank_races_per_year'] = df_200m['Avg_Races_Year_200m'].rank(ascending=False)  # More races per year is better
#df_200m['rank_sub_20_200m'] = df_200m['Sub_20_200m'].rank(ascending=False) # More Sub-20s is better 
#df_200m['rank_avg_wind_per_year'] = df_200m['Average_Wind_200m'].rank(ascending=False)  # More wind is better
df_200m['rank_avg_place_per_race'] = df_200m['Average_Place_200m'].rank(ascending=True)  # Lower finish each race is better
#df_200m['rank_avg_place_per_final'] = df_200m['Avg_Place_200m_Final_Heat'].rank(ascending=True)  # Lower finish each race is better
#df_200m['rank_avg_clutch_finish'] = df_200m['Avg_Clutch_Finish_200mF'].rank(ascending=True)  # Lower finish each race is better
#df_200m['rank_total_WA_score'] = df_200m['Total_WA_Score_200m'].rank(ascending=False)  # Higher Score is better
df_200m['rank_avg_WA_score'] = df_200m['Avg_WA_Score_200m'].rank(ascending=False)  # Higher Score is better
#df_200m['rank_total_medals'] = df_200m['total_medals'].rank(ascending=False)  # More medalds is better
#df_200m['rank_medal_score'] = df_200m['medal_score'].rank(ascending=False)  # Higher Score is better
#df_200m['Avg_Log_Cat_Score_200m'] = df_200m['Avg_Log_Cat_Score_200m'].rank(ascending=False)  # Higher Score is better
#df_200m['Avg_Lin_Cat_Score_200m'] = df_200m['Avg_Lin_Cat_Score_200m'].rank(ascending=False)  # Higher Score is better

#Final ranking
df_200m['final_rank'] = df_200m[['rank_personal_best','rank_average_sb', 'rank_career_avg',  'rank_consistency', 'rank_years_competed','rank_total_races','rank_races_per_year','rank_avg_place_per_race','rank_avg_WA_score']].mean(axis=1)

In [ ]:
df_200m.sort_values('final_rank').head(10)

In [ ]:
Final_Rankings_200m = df_200m.sort_values('final_rank')

In [ ]:
Final_Rankings_200m = pd.DataFrame(Final_Rankings_200m)
Final_Rankings_200m.head(3)

In [ ]:
#Let's make the entire dataframe a figure factory table


fig =  ff.create_table(Final_Rankings_200m)
fig.show()

#fig.write_html("Final_Rankings_200m.html")
#fig.write_image("Final_Rankings_200m.svg")

Scatter Plot Visualization

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Years_Competed_200m', y='Avg_Season_Best_200m', data=df_200m, s=100, hue='final_rank', palette='coolwarm')
plt.title('Years Competed vs. Career Average 200m Time')
plt.xlabel('Years Competed')
plt.ylabel('Average 200m Time (s)')
plt.show()

#plt.savefig('SB_avg_200_vs_yrs_competed_ranked.png', format='png', dpi=300)
#plt.savefig('SB_avg_200_vs_yrs_competed_ranked.jpg', format='jpg', dpi=300)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Years_Competed_200m', y='Career_Avg_200m', data=df_200m, s=100, hue='final_rank', palette='coolwarm')
plt.title('Years Competed vs. Career Average 200m Time')
plt.xlabel('Years Competed')
plt.ylabel('Career Average 200m Time (s)')
plt.show()

#plt.savefig('Career_avg_200_vs_yrs_competed_ranked.png', format='png', dpi=300)
#plt.savefig('Career_avg_200_vs_yrs_competed_ranked.jpg', format='jpg', dpi=300)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Years_Competed_200m', y='200_PR', data=df_200m, s=100, hue='final_rank', palette='coolwarm')
plt.title('Years Competed vs. 200m PR')
plt.xlabel('Years Competed')
plt.ylabel('200m PR (s)')
plt.show()

#plt.savefig('200_PR_vs_yrs_competed_ranked.png', format='png', dpi=300)
#plt.savefig('200m_PR_vs_yrs_competed_ranked.jpg', format='jpg', dpi=300)

Interactive Scatter Plot

In [ ]:
# @title Years Competed vs. Career Average 200m Time

df_200m = px.data.iris()
fig = px.scatter(df_200m, x="Years_Competed_200m", y="Avg_Season_Best_200m", color="final_rank",
                 size='Years_Competed_200m', hover_data=['Avg_Season_Best_200m']) #Potentially switch out career average for personal record regarding hover data.
fig.show()

#fig.write_html("avg_200_vs_yrs_competed_ranked.html")
#fig.write_image("avg_200_vs_yrs_competed_ranked.svg")

In [ ]:
# @title Years Competed vs. Career Average 200m Time

df_200m = px.data.iris()
fig = px.scatter(df_200m, x="Years_Competed_200m", y="Career_Avg_200m", color="final_rank",
                 size='Years_Competed_200m', hover_data=['Career_Avg_200m']) #Potentially switch out career average for personal record regarding hover data.
fig.show()

#fig.write_html("Career_avg_200_vs_yrs_competed_ranked.html")
#fig.write_image("Career_avg_200_vs_yrs_competed_ranked.svg")

In [ ]:
# @title Years Competed vs. 200m PR

df_200m = px.data.iris()
fig = px.scatter(df_200m, x="Years_Competed_200m", y="200_PR", color="final_rank",
                 size='Years_Competed_200m', hover_data=['200_PR'])
fig.show()

#fig.write_html("200m_PR_vs_yrs_competed_ranked.html")
#fig.write_image("200m_PR_vs_yrs_competed_ranked.svg")

In [ ]:
print(df_200m[['Athlete', 'Avg_Season_Best_200m', 'Years_Competed_200m','200_PR', 'residuals', 'z_combined', 'efficiency_score','efficiency_score_sb', 'efficiency_score_pr' 'final_rank']])

In [ ]:
df_200m_stat_analysis = df_200m[['Athlete', 'Avg_Season_Best_200m', 'Years_Competed_200m',' 200_PR', 'residuals', 'z_combined', 'efficiency_score','efficiency_score_sb', 'efficiency_score_pr' 'final_rank']]

In [ ]:
df_200m_stat_analysis = df_200m_stat_analysis.sort_values(by='final_rank')
df_200m_stat_analysis.head(3)

In [ ]:
#What sample size of the dataframe to we want to make into a figure factory table
df_200m_stat_analysis_sample = df_200m_stat_analysis[1:10]

#Cusomize Colors (Add colorscale=colorscale in parentheses of ff.create table)
#colorscale = [[0, '#4d004c'],[.5, '#f2e5ff'],[1, '#ffffff']]
#Cusomize Font Colors (Add font_colors=font in parentheses of ff.create table)
#font=['#FCFCFC', '#00EE00', '#008B00', '#004F00', '#660000', '#CD0000', '#FF3030']

table_data = df_200m_stat_analysis


fig =  ff.create_table(df_200m_stat_analysis_sample)
fig.show()

#fig.write_html("df_200m_stat_analysis_sample_ff.html")
#fig.write_image("df_200m_stat_analysis_sample_ff.svg")


In [ ]:
fig =  ff.create_table(df_200m_stat_analysis)
fig.show()

#fig.write_html("df_200m_stat_analysis_ff.html")
#fig.write_image("df_200m_stat_analysis_ff.svg")

In [ ]:
fig =  ff.create_table(df_200m)
fig.show()

#fig.write_html("df_200m_ff.html")
#fig.write_image("df_200m_ff.svg")

In [ ]:
df_200m = pd.DataFrame(df_200m)

In [ ]:
df_200m_stat_analysis = pd.DataFrame(df_200m_stat_analysis)

# **Let's put all the data frames created into an excel workbook**

In [ ]:
xlwriter = pd.ExcelWriter('200M_Analysis.xlsx')
df_200m.to_excel(xlwriter, sheet_name='200m')
Z_Score_Ranking_200m.to_excel(xlwriter, sheet_name='200m Z-Score Rankings')
Final_Rankings_200m.to_excel(xlwriter, sheet_name='200m Final Rankings')
medals_200m.to_excel(xlwriter, sheet_name='200m Athlete Medal Summary')
#df_200m_stat_analysis.to_excel(xlwriter, sheet_name='200m Statisitcal Analysis')
#athlete_category_summary_200m.to_excel(xlwriter, sheet_name='')
#athlete_category_year_summary_200m.to_excel(xlwriter, sheet_name='')
#athlete_year_summary_200m.to_excel(xlwriter, sheet_name='')
#country_category_summary_200m.to_excel(xlwriter, sheet_name='')
#country_year_summary_200m.to_excel(xlwriter, sheet_name='')
#continent_category_summary_200m.to_excel(xlwriter, sheet_name='')
#continent_year_summary_200m.to_excel(xlwriter, sheet_name='')
#athlete_country_year_category_200m.to_excel(xlwriter, sheet_name='')
xlwriter.close()

Let's get the age of each athlete per race

In [ ]:
import pandas as pd

# Convert to datetime
df['date'] = pd.to_datetime(df['date'])
df['DOB'] = pd.to_datetime(df['DOB'])

# Age in completed years at race date
df['age_at_race'] = (
    df['date'].dt.year - df['DOB'].dt.year
    - (
        (df['date'].dt.month < df['DOB'].dt.month)
        | (
            (df['date'].dt.month == df['DOB'].dt.month)
            & (df['date'].dt.day < df['DOB'].dt.day)
        )
    )
)